## FULL PROJECT — Chat with PDF (FREE LLM)

We will build a complete pipeline:

Upload PDF → Extract → Chunk → Embeddings → FAISS → HuggingFace LLM → Chat

No paid API

## STEP 1 — Install Everything (Run first)

In [1]:
!pip install langchain langchain-community langchain-text-splitters \
sentence-transformers faiss-cpu transformers accelerate pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## STEP 2 — Load LLM (Free GPT alternative)

We use FLAN-T5 via HuggingFace.

In [3]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=256
)

print("LLM Loaded ")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

LLM Loaded 


## STEP 3 — Upload Your PDF in Colab

Run this cell and upload any PDF

In [7]:
from google.colab import files
uploaded = files.upload()

Saving yourfile.pdf to yourfile.pdf


After upload, your PDF filename will appear.

👉 Replace "yourfile.pdf" below with your filename.

## STEP 4 — Load PDF → Extract Text

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("yourfile.pdf")
documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 3


Each page becomes a document.

## STEP 5 — Split Text into Chunks (Very Important)

LLMs cannot read huge text → we split it.

In [9]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

print("Chunks created:", len(docs))

Chunks created: 3


## STEP 6 — Convert Chunks → Embeddings

In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc.page_content for doc in docs]
embeddings = embed_model.encode(texts)

print("Embeddings ready:", len(embeddings))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings ready: 3


## STEP 7 — Store in Vector Database (FAISS)

In [11]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Vector DB Ready ")

Vector DB Ready 


## STEP 8 — Build Retriever (Search inside PDF)

In [12]:
def retrieve(query, k=3):
    query_embedding = embed_model.encode([query])
    D, I = index.search(np.array(query_embedding), k)
    return [texts[i] for i in I[0]]

In [13]:
# Test retrieval:

print(retrieve("summary"))

['Page 1: Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is transforming industries such as healthcare, finance, and education.', 'Page 3: Future of AI\nThe future of AI includes advancements in natural language processing, robotics, and ethical AI.', 'Page 2: Applications of AI\nAI is used for medical diagnosis, fraud detection, personalized learning, and much more.']


## STEP 9 — Connect Retriever + LLM (Real RAG)

This is the brain

In [14]:
def ask_pdf(question):

    context = retrieve(question)

    prompt = f"""
    Answer ONLY using the context below.

    Context:
    {context}

    Question: {question}
    """

    result = llm(prompt)[0]["generated_text"]
    return result

## STEP 10 — Test Your PDF Chatbot

```
print(ask_pdf("What is this document about?"))
print(ask_pdf("Give me summary"))
print(ask_pdf("Explain key points"))

```

You now have ChatGPT for PDFs

In [15]:
print(ask_pdf("What is this document about?"))
print(ask_pdf("Give me summary"))
print(ask_pdf("Explain key points"))


    Answer ONLY using the context below.

    Context:
    ['Page 3: Future of AI\nThe future of AI includes advancements in natural language processing, robotics, and ethical AI.', 'Page 1: Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is transforming industries such as healthcare, finance, and education.', 'Page 2: Applications of AI\nAI is used for medical diagnosis, fraud detection, personalized learning, and much more.']

    Question: What is this document about?
    

    Answer ONLY using the context below.

    Context:
    ['Page 1: Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is transforming industries such as healthcare, finance, and education.', 'Page 3: Future of AI\nThe future of AI includes advancements in natural language processing, robotics, and ethical AI.', 'Page 2: Applications of AI\nAI is used for medical diagnosis, fraud detection, personalized learning, and much more.']

    Question: Give me summary
    

    

### BONUS — Conversational Chat Memory

```
chat_history = []

def chat(message):
    response = ask_pdf(message)
    chat_history.append((message, response))
    return response
```
Test chat:

```
chat("Summarize the document")
chat("Explain more simply")

```

In [16]:
chat_history = []

def chat(message):
    response = ask_pdf(message)
    chat_history.append((message, response))
    return response

In [17]:
# Test chat:

chat("Summarize the document")
chat("Explain more simply")

"\n    Answer ONLY using the context below.\n\n    Context:\n    ['Page 1: Introduction to Artificial Intelligence\\nArtificial Intelligence (AI) is transforming industries such as healthcare, finance, and education.', 'Page 2: Applications of AI\\nAI is used for medical diagnosis, fraud detection, personalized learning, and much more.', 'Page 3: Future of AI\\nThe future of AI includes advancements in natural language processing, robotics, and ethical AI.']\n\n    Question: Explain more simply\n    "

### What You Built (Industry Level)

| Component    | Tool                  |
| ------------ | --------------------- |
| PDF reader   | LangChain             |
| Chunking     | Text Splitter         |
| Embeddings   | Sentence Transformers |
| Vector DB    | FAISS                 |
| LLM          | HuggingFace           |
| Architecture | RAG                   |

This is used in:

- Legal AI
- Research tools
- Company knowledge bots
- Study assistants